# Can a grader adapt to a new camera without labels?

Takes the 10 checkpoints from `retrain_and_export.ipynb` (5 seeds × CE / EBL), adapts each one to APTOS using
**500 unlabeled APTOS images**, and scores it on the **other 3,162**. Four methods:

| method | what changes | objective |
|---|---|---|
| `source` | nothing | baseline |
| `adabn` | normalization statistics only | none (re-estimated on the new camera) |
| `tent` | normalization weights | lower prediction entropy |
| `tea` | normalization weights | lower the energy of real images, raise it on Langevin samples (TEA, CVPR 2024) |

Watch the `share_pred*` columns: collapse toward grade 0 is the known failure mode under grade skew.
Any GPU runtime works. Needs `ebl_fundus-main.zip` and `runs_event/` in `MyDrive/ebl_fundus/` (from the last notebook).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ebl_fundus'
!unzip -q -o {D}/ebl_fundus-main.zip -d /content
%cd /content/ebl_fundus-main

APTOS images, resized once and cached in Drive (about 1 GB) so later sessions skip the download.
First time only: Kaggle token in Colab Secrets as `KAGGLE_API_TOKEN`, competition rules accepted.

In [ ]:
import os, sys, torch, pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, 'src')
from dataset import get_transforms

CACHE = f'{D}/aptos_224.pt'
if not os.path.exists(CACHE):
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    !pip install -q -U kaggle
    !kaggle competitions download -c aptos2019-blindness-detection -p /content/aptos
    !unzip -q -o /content/aptos/aptos2019-blindness-detection.zip "train.csv" "train_images/*" -d /content/aptos
    labels, tf = pd.read_csv('/content/aptos/train.csv'), get_transforms('test')
    load_img = lambda i: tf(Image.open(f'/content/aptos/train_images/{i}.png').convert('RGB')).half()
    with ThreadPoolExecutor(8) as ex:
        Xa = torch.stack(list(ex.map(load_img, labels.id_code)))
    torch.save((Xa, torch.as_tensor(labels.diagnosis.values)), CACHE)
Xa, ya = torch.load(CACHE)
ya = ya.numpy()
print(Xa.shape, 'grade counts', pd.Series(ya).value_counts().sort_index().tolist())

The four methods. Only normalization layers ever change (about 1% of ResNet-18's parameters).

In [ ]:
import glob, time, numpy as np, torch.nn as nn
from sklearn.metrics import cohen_kappa_score
from model import build_model
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def load(path):
    ck = torch.load(path, map_location=dev, weights_only=False)
    m = build_model(5, pretrained=False).to(dev)
    m.load_state_dict(ck['model_state'])
    return m.eval(), {'softmax': 'CE', 'ebm_margin': 'EBL'}[ck['args']['loss']], ck['args']['seed']

def batches(X, bs=64, shuffle=False):
    idx = torch.randperm(len(X)) if shuffle else torch.arange(len(X))
    for i in range(0, len(X), bs):
        yield X[idx[i:i + bs]].to(dev).float()

@torch.no_grad()
def predict(m, X):
    m.eval()
    return torch.cat([m(x).float().cpu() for x in batches(X)])

energy = lambda z: -torch.logsumexp(z, 1)
bn_layers = lambda m: [l for l in m.modules() if isinstance(l, nn.BatchNorm2d)]

@torch.no_grad()
def reestimate_bn(m, X):                       # AdaBN: normalization statistics of the new camera
    for l in bn_layers(m):
        l.reset_running_stats(); l.momentum = None
    m.train(); [m(x) for x in batches(X)]; m.eval()

def sgld(m, x, steps=20, lr=1.0, std=0.01):    # Langevin samples the model currently finds likely
    x = x.clone().requires_grad_(True)
    for _ in range(steps):
        g, = torch.autograd.grad(energy(m(x)).sum(), x)
        x = (x - lr * g + std * torch.randn_like(x)).detach().requires_grad_(True)
    return x.detach()

def adapt(m, X, method, epochs=1, lr=1e-3, sgld_steps=20):
    reestimate_bn(m, X)
    if method == 'adabn':
        return m
    for p in m.parameters():
        p.requires_grad_(False)
    params = [p for l in bn_layers(m) for p in (l.weight, l.bias)]
    for p in params:
        p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    buffer = torch.rand(len(X), *X.shape[1:]) * 4 - 2          # negatives start as noise
    for _ in range(epochs):
        for x in batches(X, 32, shuffle=True):
            m.train()
            z = m(x)
            if method == 'tent':
                loss = -(z.softmax(1) * z.log_softmax(1)).sum(1).mean()
            else:                                                # tea
                i = torch.randint(len(buffer), (len(x),))
                neg = sgld(m, buffer[i].to(dev), steps=sgld_steps)
                buffer[i] = neg.cpu()
                loss = energy(z).mean() - energy(m(neg)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    reestimate_bn(m, X)
    return m

def metrics(z, y):
    p = z.softmax(1).numpy(); pred, conf = p.argmax(1), p.max(1)
    ok, b = pred == y, np.minimum((conf * 15).astype(int), 14)
    ece = sum(abs(ok[b == k].mean() - conf[b == k].mean()) * (b == k).mean() for k in range(15) if (b == k).any())
    return dict(acc=ok.mean(), qwk=cohen_kappa_score(y, pred, weights='quadratic'), ece=ece,
                conf_minus_acc=conf.mean() - ok.mean(),
                referable_sens=(pred[y >= 2] >= 2).mean(), referable_spec=(pred[y <= 1] <= 1).mean(),
                **{f'share_pred{k}': (pred == k).mean() for k in range(5)})

Run every checkpoint × method. The labels of the 500 adaptation images are never used.

In [ ]:
N_ADAPT = 500
perm = np.random.default_rng(0).permutation(len(Xa))
A, E, yE = Xa[perm[:N_ADAPT]], Xa[perm[N_ADAPT:]], ya[perm[N_ADAPT:]]

rows = []
for path in sorted(glob.glob(f'{D}/runs_event/*/best.pt')):
    for method in ['source', 'adabn', 'tent', 'tea']:
        m, model, seed = load(path)
        torch.manual_seed(seed); t0 = time.time()
        if method != 'source':
            adapt(m, A, method)
        rows.append(dict(model=model, seed=seed, method=method, secs=round(time.time() - t0),
                         **metrics(predict(m, E), yE)))
        r = rows[-1]
        print(f"{model} {seed} {method:6s} acc {r['acc']:.3f} qwk {r['qwk']:.3f} ece {r['ece']:.3f} "
              f"grade-0 share {r['share_pred0']:.2f}  ({r['secs']}s)")
df = pd.DataFrame(rows)
df.to_csv(f'{D}/tta_results.csv', index=False)

In [ ]:
cols = ['acc', 'qwk', 'ece', 'conf_minus_acc', 'referable_sens', 'referable_spec'] + [f'share_pred{k}' for k in range(5)]
summary = df.groupby(['model', 'method'])[cols].mean()
print(summary.round(3).to_string())
print('\ntrue grade shares on the evaluation images:', np.round(np.bincount(yE, minlength=5) / len(yE), 3))

## Sweep: how many unlabeled photos does a new camera need?

Run the setup cells above (Drive, APTOS cache, methods), skip the 500-image run, then run this cell.
Every adaptation size is scored on the same 2,662 images, and all logits are saved to Drive (`tta_logits/`)
so the grade-mix correction can be computed later without a GPU.

In [ ]:
SIZES = [100, 250, 500, 1000]
perm = np.random.default_rng(0).permutation(len(Xa))
E2, yE2 = Xa[perm[1000:]], ya[perm[1000:]]          # same evaluation images for every size
os.makedirs(f'{D}/tta_logits', exist_ok=True)
np.save(f'{D}/tta_logits/labels.npy', yE2)

rows2 = []
for path in sorted(glob.glob(f'{D}/runs_event/*/best.pt')):
    for n in [0] + SIZES:
        for method in (['source'] if n == 0 else ['tent', 'tea']):
            m, model, seed = load(path)
            torch.manual_seed(seed)
            if n:
                adapt(m, Xa[perm[:n]], method)
            z = predict(m, E2)
            np.save(f'{D}/tta_logits/{model}_{seed}_{method}_{n}.npy', z.numpy())
            rows2.append(dict(model=model, seed=seed, method=method, n=n, **metrics(z, yE2)))
    print(model, seed, 'done')
sweep = pd.DataFrame(rows2)
sweep.to_csv(f'{D}/tta_sweep.csv', index=False)
print(sweep.groupby(['model', 'method', 'n'])[['acc', 'qwk', 'ece', 'conf_minus_acc', 'share_pred1', 'share_pred3']]
      .mean().round(3).to_string())